In [8]:
!pip install -q transformers accelerate pypdf
!pip install -U bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM
from pypdf import PdfReader
import torch, re, json
from pathlib import Path

In [4]:
from google.colab import files
from pathlib import Path
import re, json

# Option A: upload PDF from your machine
print("Upload your PDF…")
up = files.upload()  # pick your file in the dialog
pdf_name = next(iter(up.keys()))
PDF_PATH = Path(pdf_name)

# Option B: if your file is already in the workspace/Drive, set it manually:
# PDF_PATH = Path("/content/your_file.pdf")
print("Using:", PDF_PATH)

Upload your PDF…


Saving rag_base.pdf to rag_base.pdf
Using: rag_base.pdf


In [5]:
from pypdf import PdfReader

def pdf_to_text(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n\n".join((p.extract_text() or "") for p in reader.pages)

def clean_text(t: str) -> str:
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)                 # join hyphen line-breaks
    t = re.sub(r"[ \t]*\n(?!\s*\n)", " ", t)               # collapse single newlines
    t = re.sub(r"\s+\n", "\n", t)
    return t.strip()

raw = pdf_to_text(PDF_PATH)
text = clean_text(raw)

# Simple "chapter" split: split on common headings; fallback to big chunks if no headings
parts = re.split(r"(?i)(?:\n\s*(?:Kapitel|Chapter)\s+\d+\b|^\s*\d+(?:\.\d+)*\s+[^\n]+$)", text, flags=re.M)
parts = [p.strip() for p in parts if len(p.split()) > 80]  # keep only substantive parts
print(f"Segments detected: {len(parts)}")


Segments detected: 68


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16
)

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", quantization_config=bnb
)

def generate(text: str, max_new_tokens=512, temperature=0.7):
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature, do_sample=True, top_p=0.9
    )
    return tok.decode(out[0], skip_special_tokens=True)


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`